## Predictive Modeling

This notebook builds a model to predict whether a client will subscribe 
to a term deposit, based on the cleaned and preprocessed dataset.

**Note on `duration`:** although duration is one of the strongest predictor in the dataset, 
I have excluded it from the model, whose purpose is to answer this question: "Is it worth calling or contacting this specific customer?".
Including this variable would create a paradox: at the time of prediction, the call has not yet taken palces, making that data unavailable in a real-world scenario.
We know that longer calls (after a certain duration) have diminishing return anyway.

The goal is a model that is not just accurate, but actually realistic.

**Models:** XGBoost will be the ideal model its ability to handle imbalanced datasets and non-linear relationships without extensive preprocessing. But I will start with Logistic Regression (as baseline)  Random Forest.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, precision_score, recall_score, f1_score, roc_auc_score

In [2]:
# Load model-ready dataset
df=pd.read_csv('../data/processed_data/bank_model.csv')
print(f"Shape: {df.shape}")
df.head()

Shape: (41172, 21)


,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,campaign,previously_contacted,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,...,1,0,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,...,1,0,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
2,37,services,married,high.school,no,yes,no,telephone,may,mon,...,1,0,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,...,1,0,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
4,56,services,married,high.school,no,no,yes,telephone,may,mon,...,1,0,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no


In [3]:
# Drop the duration column
df_new= df.drop(columns=['duration'])
print(f"Shape: {df_new.shape}")
df_new.head()

Shape: (41172, 20)


,age,job,marital,education,default,housing,loan,contact,month,day_of_week,campaign,previously_contacted,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,1,0,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,1,0,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
2,37,services,married,high.school,no,yes,no,telephone,may,mon,1,0,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,1,0,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
4,56,services,married,high.school,no,no,yes,telephone,may,mon,1,0,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no


In [4]:
# Split data into X and y
X=df_new.drop(columns=['y'])

y=df_new['y']

In [5]:
X

,age,job,marital,education,default,housing,loan,contact,month,day_of_week,campaign,previously_contacted,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,1,0,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,1,0,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0
2,37,services,married,high.school,no,yes,no,telephone,may,mon,1,0,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,1,0,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0
4,56,services,married,high.school,no,no,yes,telephone,may,mon,1,0,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
41167,73,retired,married,professional.course,no,yes,no,cellular,nov,fri,1,0,0,nonexistent,-1.1,94.767,-50.8,1.028,4963.6
41168,46,blue-collar,married,professional.course,no,no,no,cellular,nov,fri,1,0,0,nonexistent,-1.1,94.767,-50.8,1.028,4963.6
41169,56,retired,married,university.degree,no,yes,no,cellular,nov,fri,2,0,0,nonexistent,-1.1,94.767,-50.8,1.028,4963.6
41170,44,technician,married,professional.course,no,no,no,cellular,nov,fri,1,0,0,nonexistent,-1.1,94.767,-50.8,1.028,4963.6


In [6]:
y

0         no
1         no
2         no
3         no
4         no
        ... 
41167    yes
41168     no
41169     no
41170    yes
41171     no
Name: y, Length: 41172, dtype: str

In [7]:
# Split data in train and test sets
np.random.seed(42)

X_train, X_test, y_train, y_test= train_test_split(X, y,
                                                   test_size=0.2)
                                                   

In [8]:
print(f"X_train length: {len(X_train)}")
print(f"X_test length: {len(X_test)}")
print(f"\ny_train length: {len(y_train)}")
print(f"y_test length: {len(y_test)}")

X_train length: 32937
X_test length: 8235

y_train length: 32937
y_test length: 8235


## Encoding

The encoding strategy was chosen based on the nature of each feature and the type of information it represents.

All categorical variables in the dataset are **nominal features** (no natural order - although `month` and `day_of_week` have an order, this is a temporal order so they will be treated as nominal categorical variables to not introduce misleading numerical relationships between thse categories).  

Because of this, **One-Hot Encoding** was selected as the main encoding technique.

 Only for the target (binary), I will use **manual mapping** to encode it.


In [9]:
## Encoding for target
y_train_encoded= y_train.map({
    'no': 0,
    'yes': 1})

y_test_encoded= y_test.map({
    'no': 0,
    'yes': 1})

# I named these with 'encoded' attribute to mantain consistency with the name given to the feature ('X_train_encoded' and 'X_test_encoded')

## One-Hot Encoding

In [10]:
cat_cols= df_new.select_dtypes(include='str').columns.tolist()
cat_cols

['job',
 'marital',
 'education',
 'default',
 'housing',
 'loan',
 'contact',
 'month',
 'day_of_week',
 'poutcome',
 'y']

In [11]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

In [12]:
preprocessor= ColumnTransformer(
    transformers=[
        ('encoding',
         OneHotEncoder(handle_unknown='ignore'),
         cat_cols)
    ],
    remainder= 'passthrough')

In [13]:
cat_cols.remove('y')
cat_cols

['job',
 'marital',
 'education',
 'default',
 'housing',
 'loan',
 'contact',
 'month',
 'day_of_week',
 'poutcome']

In [14]:
# Fit_transform on X_train
X_train_encoded= preprocessor.fit_transform(X_train)

# Transform on X test
X_test_encoded= preprocessor.transform(X_test)

In [15]:
X_train_encoded[:2]

array([[ 1.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,
         0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,
         0.0000e+00,  0.0000e+00,  0.0000e+00,  1.0000e+00,  0.0000e+00,
         0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  1.0000e+00,
         0.0000e+00,  1.0000e+00,  0.0000e+00,  1.0000e+00,  0.0000e+00,
         1.0000e+00,  0.0000e+00,  0.0000e+00,  1.0000e+00,  0.0000e+00,
         0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,
         1.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,
         0.0000e+00,  1.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,
         1.0000e+00,  0.0000e+00,  3.5000e+01,  1.0000e+00,  0.0000e+00,
         0.0000e+00,  1.1000e+00,  9.3994e+01, -3.6400e+01,  4.8600e+00,
         5.1910e+03],
       [ 0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,
         0.0000e+00,  0.0000e+00,  1.0000e+00,  0.0000e+00,  0.0000e+00,
         0.0000e+00,  0.0000e

In [16]:
# Verify
preprocessor.get_feature_names_out()

array(['encoding__job_admin.', 'encoding__job_blue-collar',
       'encoding__job_entrepreneur', 'encoding__job_housemaid',
       'encoding__job_management', 'encoding__job_retired',
       'encoding__job_self-employed', 'encoding__job_services',
       'encoding__job_student', 'encoding__job_technician',
       'encoding__job_unemployed', 'encoding__marital_divorced',
       'encoding__marital_married', 'encoding__marital_single',
       'encoding__education_basic.4y', 'encoding__education_basic.6y',
       'encoding__education_basic.9y', 'encoding__education_high.school',
       'encoding__education_professional.course',
       'encoding__education_university.degree',
       'encoding__education_unknown', 'encoding__default_no',
       'encoding__default_unknown', 'encoding__housing_no',
       'encoding__housing_yes', 'encoding__loan_no', 'encoding__loan_yes',
       'encoding__contact_cellular', 'encoding__contact_telephone',
       'encoding__month_apr', 'encoding__month_aug',
  

In [17]:
# Visualize X_train_encoded
X_train_encoded_visual= pd.DataFrame(
    X_train_encoded,
    columns= preprocessor.get_feature_names_out())

X_train_encoded_visual.head()

,encoding__job_admin.,encoding__job_blue-collar,encoding__job_entrepreneur,encoding__job_housemaid,encoding__job_management,encoding__job_retired,encoding__job_self-employed,encoding__job_services,encoding__job_student,encoding__job_technician,...,encoding__poutcome_success,remainder__age,remainder__campaign,remainder__previously_contacted,remainder__previous,remainder__emp.var.rate,remainder__cons.price.idx,remainder__cons.conf.idx,remainder__euribor3m,remainder__nr.employed
0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,35.0,1.0,0.0,0.0,1.1,93.994,-36.4,4.860,5191.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,31.0,1.0,0.0,0.0,1.4,93.918,-42.7,4.963,5228.1
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,27.0,9.0,0.0,0.0,-1.8,92.893,-46.2,1.354,5099.1
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,34.0,1.0,0.0,0.0,1.1,93.994,-36.4,4.857,5191.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,50.0,3.0,0.0,0.0,1.1,93.994,-36.4,4.857,5191.0


## Handling Class Imbalance

The dataset is heavily imbalanced, roughly 88% of clients did not subscribe. Without addressing this, any model would learn 
to simply predict "no" for everyone and still achieve 88% accuracy, 
which is completely useless in practice.

Rather than synthetically generating new data points (SMOTE) or removing 
majority class samples (both of which alter the dataset)
I chose to work with the data as it is and inform the models directly 
about the class distribution.

For Logistic Regression and Random Forest, I use `class_weight='balanced'`, 
which automatically adjusts the penalty for misclassifying the minority class. 
For XGBoost, the equivalent parameter is `scale_pos_weight`, set to the ratio 
of negative to positive samples - telling the model how much more attention 
to pay to the minority class during training.

This approach for me is cleaner and avoids introducing 
artificial bias into the training data.

## Models

Rather than jumping straight to the most complex model, I will train and compare 
three algorithms of increasing complexity. This approach is more rigorous because
if a simpler model performs comparably to a complex one, there is little reason 
to add the extra complexity.

**1. Logistic Regression**: serves as baseline, it is the simplest possible 
classifier and sets the floor. Any model worth using should comfortably 
outperform it.

**2.Random Forest**: adds complexity through ensemble learning, useful to capture non-linear relationships that logistic regression 
cannot model.

**XGBoost** is the primary candidate beacause it handles class imbalance natively through 
`scale_pos_weight`, and consistently outperforms other algorithms on 
structured datasets with mixed feature types.

The final model selection will be based on ROC-AUC and F1-score as the primary metric.
Accuracy alone is misleading on imbalanced datasets where predicting 
"no" for everyone would already yield 88% accuracy.

In [18]:
%pip install xgboost

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [19]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import xgboost
from xgboost import XGBClassifier

In [20]:
print(xgboost.__version__)

3.2.0


In [23]:
# Calculate scale_pos_weight for XGB
# Following the XGB documentation, this is the ratio of negative instances to positive instances in the training data
negative= (y_train_encoded==0).sum()
positive= (y_train_encoded==1).sum()

scale_pos_weight= negative/positive
print(f"Scale_pos_weight: {scale_pos_weight:.2f}")

Scale_pos_weight: 7.91


In [28]:
# Put models in a dictionary
models={
    "Logistic Regression": LogisticRegression(class_weight='balanced'),
    "Random Forest": RandomForestClassifier(class_weight='balanced'),
    "XGBoost Classifier": XGBClassifier(scale_pos_weight=scale_pos_weight)}

# Function to fit, score models and plot ROC curve
def fit_score(models, X_train_encoded, X_test_encoded, y_train_encoded, y_test_encoded):
    """
    Fits and evaluates given ML models 
    using ROC-AUC and F1-score.

    Parameters:
    models: a dict of differetn ML models
    X_train_encoded: encoded training data
    X_test_encoded: encoded testing data
    y_train_encoded: encoded training labels
    y_test_encoded: encoded test labels

    Output:
    dataFrame with model performance metrics.
    """
    np.random.seed(42)
    
    # A dictionary to keep the model scores
    model_scores={}

    # Loop through the models
    for name, model in models.items():
        # Fit the model
        model.fit(X_train_encoded, y_train_encoded)
        # Predictions
        y_pred=model.predict(X_test_encoded)
        # Probabilities for ROC-AUC
        y_probs=model.predict_proba(X_test_encoded)[:,1]
        # Calculate metrics
        roc_auc=roc_auc_score(y_test_encoded, y_probs)
        f1=f1_score(y_test_encoded, y_pred)
        # Append the score to model_scores
        model_scores[name]={
            'ROC-AUC': roc_auc,
            'F1-score': f1}
    
    return pd.DataFrame(model_scores).T

In [29]:
model_scores= fit_score(models= models,
                        X_train_encoded = X_train_encoded,
                        X_test_encoded= X_test_encoded,
                        y_train_encoded= y_train_encoded,
                        y_test_encoded= y_test_encoded)
model_scores

C:\Users\Windows10\Desktop\customer-churn-analysis\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


,ROC-AUC,F1-score
Logistic Regression,0.801552,0.455678
Random Forest,0.788287,0.365169
XGBoost Classifier,0.786110,0.478332


## NOTE:
**Logistic Regression initially did not converge within the default number of iterations (max_iter=100).**
This is common in datasets with mixed feature scales and after one-hot encoding (it increases dimensionality).

To address this:
1. I will increase max_iter to ensure the optimization algorithm has enough iterations to converge.
2. I will include StandardScaler to improve optimization stability ( using a Pipeline to structure preprocessing and modeling in a clean way).

In [30]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

In [32]:
log_reg= Pipeline([
        ('scaler', StandardScaler(with_mean=False)), #with_mean=False because One-Hot Encoding create a sparse matrix with a lot of 0
        ('model', LogisticRegression(max_iter=1000, class_weight='balanced'))])

In [33]:
models_2={
    "Logistic Regression": log_reg,
     "Random Forest": RandomForestClassifier(class_weight='balanced'),
    "XGBoost Classifier": XGBClassifier(scale_pos_weight=scale_pos_weight)}

model_scores= fit_score(models= models_2,
                        X_train_encoded = X_train_encoded,
                        X_test_encoded= X_test_encoded,
                        y_train_encoded= y_train_encoded,
                        y_test_encoded= y_test_encoded)
model_scores

,ROC-AUC,F1-score
Logistic Regression,0.806367,0.477964
Random Forest,0.788287,0.365169
XGBoost Classifier,0.786110,0.478332
